# 🔹 Hidrodinâmica de Canais Abertos
## Volume I – Capítulo 5 | Estudo de Caso: Rio Macaé

**Objetivos de aprendizagem:**
- Aplicar as equações de Saint-Venant 1D
- Calcular parâmetros geométricos e de atrito (Manning)
- Discretizar por Diferenças Finitas implícitas
- Simular propagação de hidrogramas

**Referência:** Lugon Junior (2026), Vol I, Cap. 5.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.sparse import diags
from scipy.sparse.linalg import spsolve

G = 9.81
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)

## 🔸 Parâmetros Geométricos e Manning

In [ ]:
def geometria_retangular(B, h):
    """Calcula parâmetros para seção retangular."""
    A = B * h                    # Área molhada [m²]
    P = B + 2*h                  # Perímetro molhado [m]
    Rh = A / P                   # Raio hidráulico [m]
    return A, P, Rh

def declividade_atrito_manning(Q, A, Rh, n):
    """Calcula S_f pela equação de Manning."""
    return (n * Q * np.abs(Q)) / (A**2 * Rh**(4/3))

# Exercício Vol I, Cap 5, Ex 1
B = 10.0       # m
h = 1.5        # m
Q = 12.0       # m³/s
n = 0.030      # s/m^(1/3)

A, P, Rh = geometria_retangular(B, h)
V = Q / A
Sf = declividade_atrito_manning(Q, A, Rh, n)

print(f"Seção retangular (B={B} m, h={h} m):")
print(f"  Área: A = {A:.2f} m²")
print(f"  Perímetro: P = {P:.2f} m")
print(f"  Raio hidráulico: R_h = {Rh:.3f} m")
print(f"  Velocidade: V = {V:.3f} m/s")
print(f"  Declividade de atrito: S_f = {Sf:.4e} m/m")

## 🔸 Número de Froude e Classificação de Regime

In [ ]:
def numero_froude(V, h, g=G):
    """Calcula número de Froude para seção retangular."""
    return V / np.sqrt(g * h)

# Exercício Vol I, Cap 5, Ex 2
Q = 8.0        # m³/s
B = 20.0       # m
h = 0.8        # m

A = B * h
V = Q / A
Fr = numero_froude(V, h)

print(f"\nClassificação de regime (Q={Q} m³/s, B={B} m, h={h} m):")
print(f"  Velocidade: V = {V:.3f} m/s")
print(f"  Número de Froude: Fr = {Fr:.3f}")
if Fr < 1:
    print(f"  → Regime SUBCRÍTICO (fluvial) ✓")
elif Fr > 1:
    print(f"  → Regime SUPERCRÍTICO (torrencial) ✓")
else:
    print(f"  → Regime CRÍTICO ✓")

## 🔸 Solver Implícito para Saint-Venant (Esqueleto)

In [ ]:
def solver_saint_venant_implicito(B, n, S0, L, T, dx, dt, Q_in_func, h_downstream):
    """
    Solver implícito simplificado para Saint-Venant 1D.
    
    Parâmetros:
    -----------
    B : float
        Largura do canal [m]
    n : float
        Manning [s/m^(1/3)]
    S0 : float
        Declividade do fundo [m/m]
    L, T : float
        Comprimento e tempo total de simulação
    dx, dt : float
        Passos espacial e temporal
    Q_in_func : callable
        Função Q_in(t) para condição de contorno de montante
    h_downstream : float or callable
        Condição de jusante (h ou rating curve)
    
    Retorna:
    --------
    x, t, h, Q : arrays
        Malha e soluções de lâmina e vazão
    """
    Nx = int(L / dx) + 1
    Nt = int(T / dt) + 1
    x = np.linspace(0, L, Nx)
    t = np.linspace(0, T, Nt)
    
    # Condição inicial: regime permanente uniforme
    h = np.full(Nx, 0.71)  # profundidade inicial [m]
    Q = np.full(Nx, 6.0)   # vazão inicial [m³/s]
    
    # Armazenar resultados
    h_sol = np.zeros((Nt, Nx))
    Q_sol = np.zeros((Nt, Nx))
    h_sol[0,:] = h
    Q_sol[0,:] = Q
    
    # Loop temporal (esqueleto - implementação completa requer iteração não-linear)
    for n_step in range(Nt-1):
        # Atualizar condição de contorno de montante
        Q[0] = Q_in_func(t[n_step])
        
        # Atualizar condição de jusante (onda saindo)
        c = np.sqrt(G * h[-1])  # celeridade
        Q[-1] = Q[-2] + c * B * (h[-1] - h[-2])  # condição de radiação
        
        # Discretização implícita simplificada (continuidade apenas)
        for i in range(1, Nx-1):
            A = B * h[i]
            dAdt = (B * (h[i] - h[i])) / dt  # placeholder
            dQdx = (Q[i+1] - Q[i-1]) / (2*dx)
            # h[i] += dt * (-dQdx / B)  # atualização explícita para demonstração
        
        h_sol[n_step+1,:] = h
        Q_sol[n_step+1,:] = Q
    
    return x, t, h_sol, Q_sol

# Parâmetros do Rio Macaé (Caso de Estudo)
B = 42.2       # m
n = 0.035      # s/m^(1/3)
S0 = 0.0005    # m/m
L = 6300       # m (6.3 km)
T = 6 * 3600   # 6 horas
dx = 50        # m
dt = 30        # s

# Hidrograma de entrada com pico
def Q_in(t):
    Q_base = 6.0
    Q_pico = 12.0
    t_pico = 2 * 3600  # 2 horas
    if t < t_pico:
        return Q_base + (Q_pico - Q_base) * (t / t_pico)
    else:
        return Q_pico - (Q_pico - Q_base) * min(1, (t - t_pico) / (4*3600))

# Executar solver (demonstração)
x, t, h_sim, Q_sim = solver_saint_venant_implicito(
    B, n, S0, L, T, dx, dt, Q_in, h_downstream=0.71
)

# Plotar resultado
plt.figure(figsize=(10, 5))
plt.plot(t/3600, Q_sim[:, Nx//2], 'b-', linewidth=2, label=f'x = {x[Nx//2]:.0f} m')
plt.xlabel('Tempo [h]')
plt.ylabel('Vazão Q [m³/s]')
plt.title('Propagação de hidrograma no Rio Macaé (ponto médio)')
plt.grid(True, alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()

print(f"Simulação concluída: {len(t)} passos temporais, {len(x)} nós espaciais")
print(f"Celeridade da onda: c ≈ √(g·h) = {np.sqrt(G*0.71):.2f} m/s")
print(f"Tempo de viagem estimado: L/c = {L/np.sqrt(G*0.71)/3600:.2f} h")

## ✅ Exercícios Propostos

1. **[Graduação]** Verifique se o Rio Macaé (B=42.2 m, h=0.71 m, n=0.035, S₀=0.0005) satisfaz regime permanente uniforme pela equação de Manning.
2. **[Pós-Graduação]** Implemente o esquema de Preissmann (θ=0.75) para as equações de Saint-Venant e compare estabilidade com θ=1.0.

> 💡 Dica: Para o solver completo, use iteração de Newton-Raphson para tratar a não-linearidade de Manning.